# 02 — ML Baseline: Late Delivery Prediction

## Objectif

Ce notebook entraîne un premier modèle de machine learning pour prédire si une commande sera livrée en retard.

Le modèle utilise le fichier `ml_dataset.csv` généré automatiquement par le pipeline Gold.

Objectif de cette étape :

- charger le dataset ML 
- vérifier la target 
- séparer les features et la target 
- entraîner un premier modèle baseline 
- évaluer ses performances 
- sauvegarder le modèle

In [20]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import joblib

Récupération du dernier dossier Gold généré par lr pipeline.

In [3]:
PROJECT_ROOT = Path("..")
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

gold_batches = [
    path for path in GOLD_DIR.iterdir()
    if path.is_dir() and path.name.startswith("gold_")
]

latest_gold_batch = max(gold_batches, key=lambda path: path.name)

ml_path = latest_gold_batch / "ml_dataset.csv"

ml_path

WindowsPath('../data/gold/gold_20260605_114542/ml_dataset.csv')

In [4]:
df = pd.read_csv(ml_path)

df.head()

,order_id,late_delivery,customer_state,main_seller_state,main_product_category,item_count,product_count,seller_count,total_price,total_freight,payment_value,payment_installments,payment_type,estimated_delivery_days,purchase_month,purchase_dayofweek
0,e481f51cbdc54678b7cc49136f2d6af7,False,SP,SP,housewares,1.0,1.0,1.0,29.99,8.72,38.71,1.0,voucher,15,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,False,BA,SP,perfumery,1.0,1.0,1.0,118.70,22.76,141.46,1.0,boleto,19,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,False,GO,SP,auto,1.0,1.0,1.0,159.90,19.22,179.12,3.0,credit_card,26,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,False,RN,MG,pet_shop,1.0,1.0,1.0,45.00,27.20,72.20,1.0,credit_card,26,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,False,SP,SP,stationery,1.0,1.0,1.0,19.90,8.72,28.62,1.0,credit_card,12,2,1


In [5]:
df.shape

(96470, 16)

In [6]:
df["late_delivery"].value_counts()

late_delivery
False    88644
True      7826
Name: count, dtype: int64

In [7]:
df["late_delivery"].value_counts(normalize=True).mul(100).round(2)

late_delivery
False    91.89
True      8.11
Name: proportion, dtype: float64

il y a beaucoup plus de commandes non en retard que de commandes en retard.
91.89 % = commandes non en retard
8.11 % = commandes en retard

In [8]:
target = "late_delivery"

X = df.drop(columns=["order_id", target])
y = df[target]

X.head()

,customer_state,main_seller_state,main_product_category,item_count,product_count,seller_count,total_price,total_freight,payment_value,payment_installments,payment_type,estimated_delivery_days,purchase_month,purchase_dayofweek
0,SP,SP,housewares,1.0,1.0,1.0,29.99,8.72,38.71,1.0,voucher,15,10,0
1,BA,SP,perfumery,1.0,1.0,1.0,118.70,22.76,141.46,1.0,boleto,19,7,1
2,GO,SP,auto,1.0,1.0,1.0,159.90,19.22,179.12,3.0,credit_card,26,8,2
3,RN,MG,pet_shop,1.0,1.0,1.0,45.00,27.20,72.20,1.0,credit_card,26,11,5
4,SP,SP,stationery,1.0,1.0,1.0,19.90,8.72,28.62,1.0,credit_card,12,2,1


Identification des colonnes numériques et catégorielles

In [9]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)

Variables numériques : ['item_count', 'product_count', 'seller_count', 'total_price', 'total_freight', 'payment_value', 'payment_installments', 'estimated_delivery_days', 'purchase_month', 'purchase_dayofweek']
Variables catégorielles : ['customer_state', 'main_seller_state', 'main_product_category', 'payment_type']


C:\Users\gaeta\AppData\Local\Temp\ipykernel_27380\833989556.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()


Split train/test

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

Train : (77176, 14)
Test : (19294, 14)


Pipeline de preprocessing + modèle

In [11]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

Ici, class_weight="balanced" aide le modèle à ne pas ignorer la classe minoritaire True.

Entrainenent 

In [12]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['customer_state','main_seller_state','main_product_category',..., 'estimated_delivery_days','purchase_month','purchase_dayofweek']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped.

Evaluation

In [13]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.95      0.64      0.76     17729
        True       0.13      0.64      0.22      1565

    accuracy                           0.64     19294
   macro avg       0.54      0.64      0.49     19294
weighted avg       0.89      0.64      0.72     19294



In [14]:
confusion_matrix(y_test, y_pred)

array([[11271,  6458],
       [  563,  1002]])

## Conclusion — Baseline Logistic Regression

La target est fortement déséquilibrée : environ 92 % des commandes ne sont pas en retard contre 8 % en retard.

La régression logistique détecte environ 64 % des retards réels.
En revanche, la précision sur les retards est faible, il y a beaucoup de fauw positis.

Le dataset ML est exploitable et le pipeline d’entraînement fonctionne. Il faut maintenant comparer avec un Random Forest.

In [16]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

n_estimators=100 : nombre d’arbres ;
max_depth=10 : limite la complexité ;
min_samples_leaf=20 : évite un modèle trop instable ;
class_weight="balanced" : prend en compte le déséquilibre des classes ;
n_jobs=-1 : utilise plusieurs cœurs CPU.

Entrainement model

In [17]:
rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['customer_state','main_seller_state','main_product_category',..., 'estimated_delivery_days','purchase_month','purchase_dayofweek']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped.

Evaluation

In [18]:
y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

       False       0.96      0.73      0.83     17729
        True       0.17      0.63      0.27      1565

    accuracy                           0.72     19294
   macro avg       0.56      0.68      0.55     19294
weighted avg       0.89      0.72      0.78     19294



In [19]:
confusion_matrix(y_test, y_pred_rf)

array([[12960,  4769],
       [  584,   981]])

## Comparaison rapide

Régression logistique vs Random Forest 

La comparaison se fera surtout sur :

- le recall de la classe `True` ;
- la precision de la classe `True`;
- le F1-score de la classe `True`;
- le nombre de faux positifs et faux négatifs.

## Conclusion Random Forest

il détecte presque autant de vrais retards : 63 % contre 64 % ;
il fait moins de fausses alertes : 4 769 faux positifs contre 6 458 ;
sa précision sur les retards monte de 13 % à 17 % ;
son F1-score sur les retards passe de 0.22 à 0.27.

Donc pour l’instant :

Random Forest est le meilleur baseline, car il garde un bon recall tout en réduisant les faux positifs. Mais ca reste tres imparfait.

## Sauvegarde du modèle baseline

Le Random Forest est retenu comme meilleur modèle baseline, car il améliore la précision et le F1-score sur la classe `True` tout en conservant un recall proche de la régression logistique.

On sauvegarde le pipeline complet, incluant :

- le preprocessing des variables numériques ;
- le preprocessing des variables catégorielles ;
- le modèle Random Forest entraîné.

In [21]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "random_forest_late_delivery_baseline.joblib"

joblib.dump(rf_model, model_path)

model_path

WindowsPath('../models/random_forest_late_delivery_baseline.joblib')